General information

In [34]:
# Time/year/month etc. use time_period as a parameter
# everything else, use filters as a parameter

Import libraries

In [35]:
import pandas as pd
import matplotlib.pyplot as plt
import time
import requests
import os

Import class

In [36]:
import sys
sys.path.insert(0, '..')

from src.fetcher import SSBcsvFetcher

fetcher = SSBcsvFetcher()

Print information of table

In [37]:
# Print nicely formatted info
fetcher.print_table_info('12292')

INFO:src.inspector:Fetching info for table 12292



Table ID: 12292
Title: 12292: Omsorgstjenester - supplerende grunnlagstall (K) 2015-2024

Variables: region, statistikkvariabel, år
Time Unit: Annual
Period: 2015 - 2024
Last Updated: 2025-06-16T06:00:00Z
Source: Statistisk sentralbyrå



Inspect filters

In [39]:
# Get dimension details for Region
dimensions = fetcher.get_dimensions('12292')
region_values = dimensions['KOKkommuneregion0000']['values']
statistics_variable_values = dimensions['ContentsCode']['values']

# Display regions as DataFrame
region_df = pd.DataFrame(list(region_values.items()), columns=['Code', 'Name'])
print("Regions:")
print(region_df.to_string())

# Display statistics variables as DataFrame
print("\n\nStatistics Variables:")
stats_df = pd.DataFrame(list(statistics_variable_values.items()), columns=['Code', 'Name'])
print(stats_df.to_string())

INFO:src.api:Fetching dimensions for table 12292


Regions:
      Code                                                Name
0     3101                                              Halden
1     3103                                                Moss
2     3105                                           Sarpsborg
3     3107                                         Fredrikstad
4     3110                                              Hvaler
5     3112                                                Råde
6     3114                                     Våler (Østfold)
7     3116                                            Skiptvet
8     3118                                       Indre Østfold
9     3120                                           Rakkestad
10    3122                                              Marker
11    3124                                             Aremark
12    3201                                               Bærum
13    3203                                               Asker
14    3205                                    

Klass check

In [ ]:
# Fetch counties from KLASS API
counties_df = fetcher.get_counties()
print("Counties from KLASS API 2026:")
print(counties_df.to_string())

# Get just the codes for filtering
county_codes = fetcher.get_county_codes()
print(f"\nCounty codes: {county_codes}")

# Get code -> name mapping
county_mapping = fetcher.get_county_mapping()
print(f"\nCounty mapping: {county_mapping}")

INFO:src.klass:Fetching classification 104 from KLASS API
INFO:src.klass:Successfully fetched 16 items from classification 104
INFO:src.klass:Fetching classification 104 from KLASS API


Counties from KLASS API 2026:
   code                              name validFrom validTo
0    03                     Oslo - Oslove      None    None
1    11                          Rogaland      None    None
2    15                   Møre og Romsdal      None    None
3    18             Nordland - Nordlánnda      None    None
4    31                           Østfold      None    None
5    32                          Akershus      None    None
6    33                          Buskerud      None    None
7    34                         Innlandet      None    None
8    39                          Vestfold      None    None
9    40                          Telemark      None    None
10   42                             Agder      None    None
11   46                          Vestland      None    None
12   50           Trøndelag - Trööndelage      None    None
13   55           Troms - Romsa - Tromssa      None    None
14   56  Finnmark - Finnmárku - Finmarkku      None    None
15   99   

INFO:src.klass:Successfully fetched 16 items from classification 104
INFO:src.klass:Fetching classification 104 from KLASS API



County codes: ['03', '11', '15', '18', '31', '32', '33', '34', '39', '40', '42', '46', '50', '55', '56', '99']


INFO:src.klass:Successfully fetched 16 items from classification 104



County mapping: {'03': 'Oslo - Oslove', '11': 'Rogaland', '15': 'Møre og Romsdal', '18': 'Nordland - Nordlánnda', '31': 'Østfold', '32': 'Akershus', '33': 'Buskerud', '34': 'Innlandet', '39': 'Vestfold', '40': 'Telemark', '42': 'Agder', '46': 'Vestland', '50': 'Trøndelag - Trööndelage', '55': 'Troms - Romsa - Tromssa', '56': 'Finnmark - Finnmárku - Finmarkku', '99': 'Uoppgitt'}


Fetch DataFrame

In [ ]:
# Fetch all statistics variables from the dimensions with better error handling
# Get all statistics variable codes from dimensions
dimensions = fetcher.get_dimensions('12292')
statistics_variable_values = dimensions['ContentsCode']['values']

# Get all county codes from KLASS API
county_codes = fetcher.get_county_codes()
county_mapping = fetcher.get_county_mapping()

print(f"Found {len(county_codes)} counties")
print(f"Found {len(statistics_variable_values)} statistics variables\n")

# Create a dictionary to store dataframes with clean names
dfs = {}
failed_variables = []
skipped_variables = []

# Fetch data for each statistics variable using county codes from KLASS
for code, name in statistics_variable_values.items():
    try:
        print(f"Fetching: {code} - {name}...")
        
        df = fetcher.fetch_clean_table(
            '12292',
            time_period='top(10)',
            filters={
                'KOKkommuneregion0000': ','.join(county_codes),
                'ContentsCode': code
            }
        )
        
        # Check if we got any data
        if df.empty:
            print(f"⊘ Skipped (no data): {code}\n")
            skipped_variables.append((code, name, "No data returned"))
            continue
        
        # Clean the dataframe name
        clean_name = name.replace(' ', '_').replace('(', '').replace(')', '').replace('-', '_').replace(',', '').replace('/', '_')
        clean_name = clean_name[:80]
        
        dfs[clean_name] = df
        print(f"✓ Loaded: {clean_name} - {len(df)} rows\n")
        
        # Add delay to avoid rate limiting
        time.sleep(2)
        
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 400:
            print(f"⊘ Skipped (not available): {code} - {str(e)[:60]}\n")
            skipped_variables.append((code, name, "400 Bad Request - incompatible dimensions"))
        else:
            print(f"✗ Failed to load {code}: {str(e)[:100]}\n")
            failed_variables.append((code, name))
    except Exception as e:
        print(f"✗ Failed to load {code}: {str(e)[:100]}\n")
        failed_variables.append((code, name))

print(f"\n{'='*60}")
print(f"Total dataframes loaded: {len(dfs)}")
print(f"Skipped variables (not compatible): {len(skipped_variables)}")
print(f"Failed variables (errors): {len(failed_variables)}")

if skipped_variables:
    print("\nSkipped variables (not available for these regions):")
    for code, name, reason in skipped_variables[:5]:  # Show first 5
        print(f"  - {code}: {name}")
    if len(skipped_variables) > 5:
        print(f"  ... and {len(skipped_variables) - 5} more")

if failed_variables:
    print("\nFailed variables:")
    for code, name in failed_variables[:5]:  # Show first 5
        print(f"  - {code}: {name}")
    if len(failed_variables) > 5:
        print(f"  ... and {len(failed_variables) - 5} more")

INFO:src.api:Fetching dimensions for table 12292
INFO:src.klass:Fetching classification 104 from KLASS API
INFO:src.klass:Successfully fetched 16 items from classification 104
INFO:src.klass:Fetching classification 104 from KLASS API
INFO:src.klass:Successfully fetched 16 items from classification 104
INFO:src.api:Fetching dimensions for table 12292


Found 16 counties
Found 49 statistics variables

Fetching: KOSaarsverkvg0000 - Brukerrettede årsverk med videregående helsefaglig utdanning i omsorgstjenesten (antall)...


INFO:src.api:Fetching pivot table 12292
INFO:src.api:URL: https://data.ssb.no/api/pxwebapi/v2/tables/12292/data?lang=no&outputformat=csv&outputformatparams=separatorsemicolon,usecodesandtexts&stub=Tid&heading=statistikkvariabel&valuecodes[Tid]=top%2810%29&valuecodes[KOKkommuneregion0000]=03%2C11%2C15%2C18%2C31%2C32%2C33%2C34%2C39%2C40%2C42%2C46%2C50%2C55%2C56%2C99&valuecodes[ContentsCode]=KOSaarsverkvg0000
ERROR:src.api:Error fetching pivot table: 400 Client Error: Bad Request for url: https://data.ssb.no/api/pxwebapi/v2/tables/12292/data?lang=no&outputformat=csv&outputformatparams=separatorsemicolon,usecodesandtexts&stub=Tid&heading=statistikkvariabel&valuecodes%5BTid%5D=top%2810%29&valuecodes%5BKOKkommuneregion0000%5D=03%2C11%2C15%2C18%2C31%2C32%2C33%2C34%2C39%2C40%2C42%2C46%2C50%2C55%2C56%2C99&valuecodes%5BContentsCode%5D=KOSaarsverkvg0000
INFO:src.api:Fetching dimensions for table 12292


⊘ Skipped (not available): KOSaarsverkvg0000 - 400 Client Error: Bad Request for url: https://data.ssb.no/a

Fetching: KOSaarsverkuniho0000 - Brukerrettede årsverk med høyere helsefaglig utdanning i omsorgstjenesten (antall)...


INFO:src.api:Fetching pivot table 12292
INFO:src.api:URL: https://data.ssb.no/api/pxwebapi/v2/tables/12292/data?lang=no&outputformat=csv&outputformatparams=separatorsemicolon,usecodesandtexts&stub=Tid&heading=statistikkvariabel&valuecodes[Tid]=top%2810%29&valuecodes[KOKkommuneregion0000]=03%2C11%2C15%2C18%2C31%2C32%2C33%2C34%2C39%2C40%2C42%2C46%2C50%2C55%2C56%2C99&valuecodes[ContentsCode]=KOSaarsverkuniho0000
ERROR:src.api:Error fetching pivot table: 400 Client Error: Bad Request for url: https://data.ssb.no/api/pxwebapi/v2/tables/12292/data?lang=no&outputformat=csv&outputformatparams=separatorsemicolon,usecodesandtexts&stub=Tid&heading=statistikkvariabel&valuecodes%5BTid%5D=top%2810%29&valuecodes%5BKOKkommuneregion0000%5D=03%2C11%2C15%2C18%2C31%2C32%2C33%2C34%2C39%2C40%2C42%2C46%2C50%2C55%2C56%2C99&valuecodes%5BContentsCode%5D=KOSaarsverkuniho0000
INFO:src.api:Fetching dimensions for table 12292


...

⊘ Skipped (not available): KOSlangtid0000 - 400 Client Error: Bad Request for url: https://data.ssb.no/a

Fetching: KOSbeboersykehje0000 - Institusjon - sykehjemsbeboere (antall)...


INFO:src.api:Fetching pivot table 12292
INFO:src.api:URL: https://data.ssb.no/api/pxwebapi/v2/tables/12292/data?lang=no&outputformat=csv&outputformatparams=separatorsemicolon,usecodesandtexts&stub=Tid&heading=statistikkvariabel&valuecodes[Tid]=top%2810%29&valuecodes[KOKkommuneregion0000]=03%2C11%2C15%2C18%2C31%2C32%2C33%2C34%2C39%2C40%2C42%2C46%2C50%2C55%2C56%2C99&valuecodes[ContentsCode]=KOSbeboersykehje0000
ERROR:src.api:Error fetching pivot table: 400 Client Error: Bad Request for url: https://data.ssb.no/api/pxwebapi/v2/tables/12292/data?lang=no&outputformat=csv&outputformatparams=separatorsemicolon,usecodesandtexts&stub=Tid&heading=statistikkvariabel&valuecodes%5BTid%5D=top%2810%29&valuecodes%5BKOKkommuneregion0000%5D=03%2C11%2C15%2C18%2C31%2C32%2C33%2C34%2C39%2C40%2C42%2C46%2C50%2C55%2C56%2C99&valuecodes%5BContentsCode%5D=KOSbeboersykehje0000
INFO:src.api:Fetching dimensions for table 12292


⊘ Skipped (not available): KOSbeboersykehje0000 - 400 Client Error: Bad Request for url: https://data.ssb.no/a

Fetching: KOSbialthelsum0000 - Beboere i bolig m/ fast tilknyttet bemanning hele døgnet (antall)...


INFO:src.api:Fetching pivot table 12292
INFO:src.api:URL: https://data.ssb.no/api/pxwebapi/v2/tables/12292/data?lang=no&outputformat=csv&outputformatparams=separatorsemicolon,usecodesandtexts&stub=Tid&heading=statistikkvariabel&valuecodes[Tid]=top%2810%29&valuecodes[KOKkommuneregion0000]=03%2C11%2C15%2C18%2C31%2C32%2C33%2C34%2C39%2C40%2C42%2C46%2C50%2C55%2C56%2C99&valuecodes[ContentsCode]=KOSbialthelsum0000


KeyboardInterrupt: 